In [0]:
from pyspark.sql import functions as F

bronze_sellers = spark.table("brazilian_ecommerce.prod.bronze_sellers")

staging_sellers = (
    bronze_sellers
    .drop("seller_city", "seller_state")
    .filter(F.col("seller_id").isNotNull())   # PK can't be null
    .dropDuplicates(["seller_id"])            # guarantee PK uniqueness
)

full_table_name = "brazilian_ecommerce.prod.staging_sellers"

(
    staging_sellers.write
    .mode("overwrite")
    .option("comment", "sellers staging table")
    .option("overwriteSchema", "true")
    .saveAsTable(full_table_name)
)

# enforce primary key at the table level (Unity Catalog, informational constraint)
spark.sql(f"""
    ALTER TABLE {full_table_name}
    ALTER COLUMN seller_id SET NOT NULL
""")

spark.sql(f"""
    ALTER TABLE {full_table_name}
    ADD CONSTRAINT pk_staging_sellers PRIMARY KEY (seller_id)
""")

print(f"Loaded {staging_sellers.count()} rows into {full_table_name}")